===========================================================================
# Evaluation
===========================================================================

The purpose of this notebook is to evaluate whether prompt engineering can transform the same
validated hospital-performance evidence into different communication styles without changing
the underlying facts. Due to computational limitations, all experiments use 10 facilities per
training, development, and testing split.


## Task 1: Imports and Setup
--------------------------------------------------------------------------

In [167]:
# Import necessary libraries

import json
import re
import time
import warnings
from pathlib import Path
import textwrap
import torch
import inspect

import numpy as np
import pandas as pd

from scipy.stats import kruskal
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

warnings.filterwarnings("ignore")

line = "--" * 40
double = "==" * 40



In [168]:
# Formatting for printing outputs, function repository, and insight repository
line = "--" * 40
double = "==" * 40


def print_outputs(title = None, *outputs):

    line = "--" * 40
    double = "==" * 40

    if title:
        print(f"{double}\n{title}\n{double}")
    else:
        print(line)

    for item in outputs:

        if len(item) == 2:
            subtitle, output = item
            print_line = True
        else:
            subtitle, output, print_line = item

        if subtitle:
            print(f"{subtitle}")

        print(output)

        if print_line:
            print(line)


# Storing insights inside insights repo
insights_repository = pd.DataFrame(columns = [
    "stage",
    "section",
    "experiment",
    "parameter",
    "metric",
    "value",
    "status",
    "decision",
    "insight",
    "notes"
])


def store_insight(
    insight,
    stage = None,
    section = None,
    experiment = None,
    parameter = None,
    metric = None,
    value = None,
    status = None,
    decision = None,
    notes = None):

    global insights_repository

    new_record = pd.DataFrame([{
        "stage": stage,
        "section": section,
        "experiment": experiment,
        "parameter": parameter,
        "metric": metric,
        "value": value,
        "status": status,
        "decision": decision,
        "insight": insight,
        "notes": notes
    }])

    insights_repository = pd.concat(
        [insights_repository, new_record],
        ignore_index = True
    )

    print_outputs("New Insight Added to Repository")

    if stage:
        print(f"Stage      : {stage}")
    if section:
        print(f"Section    : {section}")
    if experiment:
        print(f"Experiment : {experiment}")
    if parameter:
        print(f"Parameter  : {parameter}")
    if metric:
        print(f"Metric     : {metric}")
    if value is not None:
        print(f"Value      : {value}")
    if status:
        print(f"Status     : {status}")
    if decision:
        print(f"Decision   : {textwrap.fill(str(decision), width = 75)}")

    print(f"\nInsight:  {textwrap.fill(str(insight), width = 75)}")

    if notes:
        print(f"\nNotes: {textwrap.fill(str(notes), width = 75)}")

    print("--" * 40)


# Create function to store information in function repository
function_repository = pd.DataFrame(
    columns = ["Function", "Description", "Input", "Output"]
)


def store_function(function, description = None, input = None, output = None):

    global function_repository

    function_name = function.__name__

    new_row = {
        "Function": function_name,
        "Description": description,
        "Input": str(inspect.signature(function)),
        "Output": output
    }

    if function_name in function_repository["Function"].values:

        for column, value in new_row.items():
            function_repository.loc[
                function_repository["Function"] == function_name,
                column
            ] = value

        print_outputs(
            f"Function {function_name} has been updated in the function_repository."
        )

    else:

        function_repository = pd.concat(
            [function_repository, pd.DataFrame([new_row])],
            ignore_index = True
        )

        print_outputs(
            f"Function '{function_name}' has been added to the function_repository."
        )


# Store repository functions
store_function(
    print_outputs,
    description = "Returns notebook outputs in a consistent organized format."
)

store_function(
    store_insight,
    description = "Stores decisions, findings, and interpretations for later reporting."
)

store_function(
    store_function,
    description = "Stores reusable functions and their signatures for traceability."
)

display(function_repository)

Function 'print_outputs' has been added to the function_repository.
Function 'store_insight' has been added to the function_repository.
Function 'store_function' has been added to the function_repository.


,Function,Description,Input,Output
0,print_outputs,Returns notebook outputs in a consistent organ...,"(title=None, *outputs)",None
1,store_insight,"Stores decisions, findings, and interpretation...","(insight, stage=None, section=None, experiment...",None
2,store_function,Stores reusable functions and their signatures...,"(function, description=None, input=None, outpu...",None


## Task 2: Loading Narrative Experiment
--------------------------------------------------------------------------

In [169]:
# Load outputs from Notebook 4

data_dir = Path("Data/Narratives")

prompt_experiment = pd.read_csv(data_dir / "prompt_experiment.csv")

with open(data_dir / "communication_prompts.json", "r", encoding="utf-8") as file:
    communication_prompts = json.load(file)

prompt_experiment["Evaluation Source"] = prompt_experiment["Performance Evidence"].fillna("")

context_mask = prompt_experiment["Context Evidence"].fillna("").str.strip() != ""
prompt_experiment.loc[context_mask, "Evaluation Source"] += (
    "\n" + prompt_experiment.loc[context_mask, "Context Evidence"]
)

print_outputs(
    "Evaluation Data",
    ("Prompt Experiment Rows", len(prompt_experiment)),
    ("Facilities", prompt_experiment["Facility ID"].nunique()),
    ("Communication Styles", list(communication_prompts.keys()))
)

print("\nRows by Split")
display(prompt_experiment.groupby("Split").size().rename("Rows").reset_index())

print("\nPrompt Experiment Columns")
print(prompt_experiment.columns.tolist())

Evaluation Data
Prompt Experiment Rows
60
--------------------------------------------------------------------------------
Facilities
15
--------------------------------------------------------------------------------
Communication Styles
['Patient Friendly', 'Executive Summary', 'Clinical', 'Community Report']
--------------------------------------------------------------------------------

Rows by Split


,Split,Rows
0,Development,20
1,Testing,20
2,Training,20



Prompt Experiment Columns
['Facility ID', 'Facility Name', 'Audience', 'Prompt Version', 'Performance Evidence', 'Context Evidence', 'Prompt', 'Split', 'Evaluation Source']


## Task 3: Generation Model
--------------------------------------------------------------------------


In [170]:
# Initialize instruction model

generation_model_name = "google/flan-t5-large"

generation_tokenizer = AutoTokenizer.from_pretrained(generation_model_name)

generation_model = AutoModelForSeq2SeqLM.from_pretrained(generation_model_name)

generation_model.eval()

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


T5ForConditionalGeneration(
  (shared): Embedding(32128, 1024)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 1024)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=1024, out_features=1024, bias=False)
              (k): Linear(in_features=1024, out_features=1024, bias=False)
              (v): Linear(in_features=1024, out_features=1024, bias=False)
              (o): Linear(in_features=1024, out_features=1024, bias=False)
              (relative_attention_bias): Embedding(32, 16)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=1024, out_features=2816, bias=False)
              (wi_1): Linear(in_features=1024, out_features=2816, bias=False)
       

In [171]:
# Define generation parameters

generation_parameters = {
    "max_new_tokens": 200,
    "min_new_tokens": 40,
    "do_sample": False,
    "num_beams": 2,
    "no_repeat_ngram_size": 2,
    "early_stopping": True
}

In [172]:
# Generate text using FLAN-T5

def generate_text(
    prompt,
    max_new_tokens = 200,
    min_new_tokens = 40,
    do_sample = False,
    num_beams = 2,
    no_repeat_ngram_size = 2,
    early_stopping = True
):

    inputs = generation_tokenizer(prompt, return_tensors = "pt", truncation = True, max_length = 512)

    with torch.no_grad():
        outputs = generation_model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            min_new_tokens = min_new_tokens,
            do_sample = do_sample,
            num_beams = num_beams,
            no_repeat_ngram_size = no_repeat_ngram_size,
            early_stopping = early_stopping
        )

    generated_text = generation_tokenizer.decode(outputs[0], skip_special_tokens = True)

    return generated_text


print_outputs(
    "Generation Model",
    ("Model", generation_model_name),
    ("Generation Parameters", generation_parameters)
)

Generation Model
Model
google/flan-t5-large
--------------------------------------------------------------------------------
Generation Parameters
{'max_new_tokens': 200, 'min_new_tokens': 40, 'do_sample': False, 'num_beams': 2, 'no_repeat_ngram_size': 2, 'early_stopping': True}
--------------------------------------------------------------------------------


## Task 4: Evaluation Functions
--------------------------------------------------------------------------

Evaluation prioritizes measures that are intuitive for this application. Readability estimates
how difficult the rewritten narrative is to read. Numeric recall measures whether hospital
statistics survive generation. Direction recall checks whether terms such as above, below,
faster, and slower remain present. Compression shows how much the model shortens or expands
the factual source.


In [173]:
# Evaluating numerical values

def extract_numbers(text):
    return re.findall(r"(?<!\w)-?\d+(?:\.\d+)?", str(text))

# Did the summary preserve original numbers
def numeric_recall(source, generated):

    source_numbers = extract_numbers(source)
    generated_numbers = extract_numbers(generated)

    if not source_numbers:
        return 1.0

    remaining = generated_numbers.copy()
    matches = 0

    for number in source_numbers:
        if number in remaining:
            matches += 1
            remaining.remove(number)

    return matches / len(source_numbers)


# Did the summary generate numbers actually from the source
def numeric_precision(source, generated):
    source_numbers = extract_numbers(source)
    generated_numbers = extract_numbers(generated)

    if not generated_numbers:
        return 1.0 if not source_numbers else 0.0

    remaining = source_numbers.copy()
    matches = 0

    for number in generated_numbers:
        if number in remaining:
            matches += 1
            remaining.remove(number)

    return matches / len(generated_numbers)

In [174]:
# Evaluating direction and word syllabels

# Did the summaries preserve the classification meanings
def direction_recall(source, generated):

    direction_terms = [
        "above", "below", "faster", "slower",
        "better", "worse", "near", "average"]

    source_terms = [
        term for term in direction_terms
        if term in str(source).lower()]

    if not source_terms:
        return 1.0

    generated_lower = str(generated).lower()

    return np.mean([
        term in generated_lower
        for term in source_terms
    ])


def count_syllables(word):

    word = re.sub(r"[^a-z]", "", word.lower())

    if not word:
        return 0

    groups = re.findall(r"[aeiouy]+", word)
    syllables = max(1, len(groups))

    if word.endswith("e") and syllables > 1:
        syllables -= 1

    return max(1, syllables)

# How readable are the summaries
def flesch_reading_ease(text):

    sentences = max(1, len(re.findall(r"[.!?]+", str(text))))

    words = re.findall(r"\b[A-Za-z]+\b", str(text))

    if not words:
        return np.nan

    syllables = sum(count_syllables(word) for word in words)

    return (
        206.835
        - 1.015 * (len(words) / sentences)
        - 84.6 * (syllables / len(words)))

# Did the summaries become shorter or longer
def evaluate_generation(source, generated):

    source_words = max(1, len(str(source).split()))

    generated_words = len(str(generated).split())

    return {
        "Flesch Reading Ease": flesch_reading_ease(generated),
        "Numeric Recall": numeric_recall(source, generated),
        "Numeric Precision": numeric_precision(source, generated),
        "Direction Recall": direction_recall(source, generated),
        "Generated Words": generated_words,
        "Compression Ratio": generated_words / source_words
    }


## Task 5: Training Prompt
--------------------------------------------------------------------------

The best prompt version for each style is selected primarily by factual
preservation, with readability used as a secondary criterion for public-facing styles.


In [175]:
# Implementation: Generate training narratives

training_prompts = prompt_experiment[prompt_experiment["Split"] == "Training"].copy()

training_rows = []

for count, (_, row) in enumerate(training_prompts.iterrows(), start=1):

    start = time.perf_counter()
    output = generate_text(row["Prompt"], **generation_parameters)
    metrics = evaluate_generation(row["Evaluation Source"], output)

    training_rows.append({
        **row.to_dict(),
        "Generated Narrative": output,
        "Generation Seconds": time.perf_counter() - start,
        **metrics
    })

    print(f"Training: {count}/{len(training_prompts)}", end="\r")

training_results = pd.DataFrame(training_rows)

print_outputs(
    "Training Evaluation Complete",
    ("Generations", len(training_results)),
    ("Average Seconds", round(training_results["Generation Seconds"].mean(), 2))
)

Training Evaluation Complete
Generations
20
--------------------------------------------------------------------------------
Average Seconds
26.67
--------------------------------------------------------------------------------


In [176]:
# Summarize training performance by communication style

training_summary = training_results.groupby("Audience", as_index=False).agg({
    "Numeric Recall": "mean",
    "Numeric Precision": "mean",
    "Direction Recall": "mean",
    "Flesch Reading Ease": "mean",
    "Compression Ratio": "mean"
})

print("TRAINING RESULTS")
display(training_summary.round(3))

TRAINING RESULTS


,Audience,Numeric Recall,Numeric Precision,Direction Recall,Flesch Reading Ease,Compression Ratio
0,Clinical,0.174,0.883,0.2,22.926,0.194
1,Community Report,0.020,0.200,0.0,43.116,0.420
2,Executive Summary,0.040,0.400,0.0,37.503,0.415
3,Patient Friendly,0.000,0.000,0.2,37.539,0.376


In [ ]:
# Double Check: Identify best-performing prompt version by communication style

prompt_performance = training_results.groupby(
    ["Audience", "Prompt Version"], as_index=False
).agg({
    "Numeric Recall": "mean",
    "Numeric Precision": "mean",
    "Direction Recall": "mean",
    "Flesch Reading Ease": "mean"
})

prompt_performance["Prompt Score"] = (
    prompt_performance["Numeric Recall"] +
    prompt_performance["Numeric Precision"] +
    prompt_performance["Direction Recall"]
) / 3

best_prompts = prompt_performance.loc[
    prompt_performance.groupby("Audience")["Prompt Score"].idxmax()
].reset_index(drop=True)

print("BEST PROMPTS BY COMMUNICATION STYLE")
print("=" * 80)

display(best_prompts[[
    "Audience",
    "Prompt Version",
    "Prompt Score",
    "Numeric Recall",
    "Numeric Precision",
    "Direction Recall",
    "Flesch Reading Ease"
]].round(3))

## Task 6: Development Validation
--------------------------------------------------------------------------

Only the best training prompt for each communication style is carried into development. This reduces computation and tests whether the selected prompt behaves similarly on new set of hospitals.


In [177]:
# Implementation: Generate development narratives

development_prompts = prompt_experiment[
    prompt_experiment["Split"] == "Development"
].copy()

development_rows = []

for count, (_, row) in enumerate(development_prompts.iterrows(), start=1):

    output = generate_text(row["Prompt"], **generation_parameters)
    metrics = evaluate_generation(row["Evaluation Source"], output)

    development_rows.append({
        **row.to_dict(),
        "Generated Narrative": output,
        **metrics
    })

    print(f"Development: {count}/{len(development_prompts)}", end="\r")

development_results = pd.DataFrame(development_rows)

development_summary = development_results.groupby("Audience", as_index=False).agg({
    "Numeric Recall": "mean",
    "Numeric Precision": "mean",
    "Direction Recall": "mean",
    "Flesch Reading Ease": "mean",
    "Compression Ratio": "mean"
})

print("\nDEVELOPMENT RESULTS")
display(development_summary.round(3))

Development: 20/20
DEVELOPMENT RESULTS


,Audience,Numeric Recall,Numeric Precision,Direction Recall,Flesch Reading Ease,Compression Ratio
0,Clinical,0.176,1.0,0.0,40.427,0.199
1,Community Report,0.111,0.6,0.0,44.899,0.351
2,Executive Summary,0.000,0.0,0.0,28.543,0.340
3,Patient Friendly,0.022,0.2,0.0,41.654,0.350


In [178]:
# Double Check: Display one development facility across communication styles

example_facility = development_results["Facility ID"].iloc[0]
example_results = development_results[
    development_results["Facility ID"] == example_facility
].copy()

print("DEVELOPMENT EXAMPLE")
print("=" * 100)
print("Facility:", example_results["Facility Name"].iloc[0])

for _, row in example_results.iterrows():
    print("\n" + "=" * 100)
    print("AUDIENCE:", row["Audience"])
    print("-" * 100)
    print(row["Generated Narrative"])

DEVELOPMENT EXAMPLE
Facility: GOOD SAMARITAN HOSPITAL

AUDIENCE: Patient Friendly
----------------------------------------------------------------------------------------------------
Good Samariah Hospital is a hospital that provides timely and effective care. The hospital has weaknesses in healthcare Associated Infections, such as Clostridium Difficile and CLABSI.

AUDIENCE: Executive Summary
----------------------------------------------------------------------------------------------------
The facility is a Good Samaritan Hospital. The strengths of the hospital are timely and effective care. Healthcare-Associated Infections are weak. Overall, the facility performs well.

AUDIENCE: Clinical
----------------------------------------------------------------------------------------------------
Healthcare-Associated Infections (CLABSI) in ICU + select wards was 2230.0 device days. The CLABsI was 3.0 cases. Overall, the hospital performed well.

AUDIENCE: Community Report
-----------------

## Task 7: Final Testing
--------------------------------------------------------------------------

The selected prompts are finally applied to 5 held-out testing hospitals. No prompt selection
is performed on this split. These results represent the final performance of the communication
system.


In [179]:
# Implementation: Generate final held-out testing narratives

testing_prompts = prompt_experiment[prompt_experiment["Split"] == "Testing"].copy()

testing_rows = []

for count, (_, row) in enumerate(testing_prompts.iterrows(), start=1):

    output = generate_text(row["Prompt"], **generation_parameters)
    metrics = evaluate_generation(row["Evaluation Source"], output)

    testing_rows.append({
        **row.to_dict(),
        "Generated Narrative": output,
        **metrics
    })

    print(f"Testing: {count}/{len(testing_prompts)}", end="\r")

testing_results = pd.DataFrame(testing_rows)

testing_summary = testing_results.groupby("Audience", as_index=False).agg({
    "Numeric Recall": "mean",
    "Numeric Precision": "mean",
    "Direction Recall": "mean",
    "Flesch Reading Ease": "mean",
    "Compression Ratio": "mean"
})

print("\nFINAL TESTING RESULTS")
display(testing_summary.round(3))

Testing: 20/20
FINAL TESTING RESULTS


,Audience,Numeric Recall,Numeric Precision,Direction Recall,Flesch Reading Ease,Compression Ratio
0,Clinical,0.191,0.933,0.0,34.861,0.182
1,Community Report,0.044,0.200,0.4,34.291,0.362
2,Executive Summary,0.153,0.400,0.6,33.823,0.511
3,Patient Friendly,0.044,0.300,0.4,35.090,0.342


## Task 8: Statistical and Plain-Language Interpretation
--------------------------------------------------------------------------

Because only 5 hospitals are included in each split, statistical tests are interpreted
cautiously. The final table emphasizes effect direction and practical meaning rather than
p-values alone.


In [184]:
# Compare communication styles on held-out testing sample

metric_rows = []

for metric in [
    "Flesch Reading Ease",
    "Numeric Recall",
    "Numeric Precision",
    "Direction Recall",
    "Compression Ratio"
]:

    groups = [
        group[metric].dropna().values
        for _, group in testing_results.groupby("Audience")
    ]

    statistic, p_value = kruskal(*groups)

    metric_rows.append({
        "Metric": metric,
        "Kruskal-Wallis H": statistic,
        "p-value": p_value,
        "Significant at 0.05": p_value < 0.05
    })

statistical_results = pd.DataFrame(metric_rows)

display(statistical_results.round(4))

interpretation = testing_summary.copy()
interpretation["Numeric Preservation"] = (
    interpretation["Numeric Recall"] * 100
).round(1).astype(str) + "%"

interpretation["Numeric Accuracy"] = (
    interpretation["Numeric Precision"] * 100
).round(1).astype(str) + "%"

interpretation["Direction Preservation"] = (
    interpretation["Direction Recall"] * 100
).round(1).astype(str) + "%"

interpretation["Reading Ease"] = interpretation["Flesch Reading Ease"].round(1)
interpretation["Relative Length"] = interpretation["Compression Ratio"].round(2)

display(interpretation[[
    "Audience",
    "Numeric Preservation",
    "Numeric Accuracy",
    "Direction Preservation",
    "Reading Ease",
    "Relative Length"
]])

,Metric,Kruskal-Wallis H,p-value,Significant at 0.05
0,Flesch Reading Ease,0.2343,0.9719,False
1,Numeric Recall,7.2831,0.0634,False
2,Numeric Precision,6.5208,0.0888,False
3,Direction Recall,3.9670,0.2650,False
4,Compression Ratio,14.5543,0.0022,True


,Audience,Numeric Preservation,Numeric Accuracy,Direction Preservation,Reading Ease,Relative Length
0,Clinical,19.1%,93.3%,0.0%,34.9,0.18
1,Community Report,4.4%,20.0%,40.0%,34.3,0.36
2,Executive Summary,15.3%,40.0%,60.0%,33.8,0.51
3,Patient Friendly,4.4%,30.0%,40.0%,35.1,0.34


## Task 9: Example Outputs
--------------------------------------------------------------------------


In [185]:
# Display one testing hospital across all communication styles

example_id = testing_results.iloc[0]["Facility ID"]

example_outputs = testing_results[
    testing_results["Facility ID"] == example_id
][[
    "Audience",
    "Generated Narrative",
    "Numeric Recall",
    "Numeric Precision",
    "Direction Recall",
    "Flesch Reading Ease"
]]

print("FACILITY")
print("-" * 80)
print(testing_results[testing_results["Facility ID"] == example_id].iloc[0]["Facility Name"])

display(example_outputs)

FACILITY
--------------------------------------------------------------------------------
FLAGSTAFF MEDICAL CENTER


,Audience,Generated Narrative,Numeric Recall,Numeric Precision,Direction Recall,Flesch Reading Ease
0,Patient Friendly,The facility FLAGSTAFF MEDICAL CENTER has bett...,0.111111,0.5,1.0,48.658571
1,Executive Summary,The facility FLAGSTAFF MEDICAL CENTER has a st...,0.000000,0.0,1.0,36.084608
2,Clinical,Healthcare-Associated Infections - CLABSI (ICU...,0.272727,1.0,0.0,47.352381
3,Community Report,FLAGSTAFF MEDICAL CENTER has a better than the...,0.222222,1.0,1.0,40.799038


In [186]:
# Display complete generated narratives

for _, row in testing_results[testing_results["Facility ID"] == example_id].iterrows():
    print("\n" + "=" * 100)
    print(row["Audience"].upper())
    print("-" * 100)
    print(row["Generated Narrative"])


PATIENT FRIENDLY
----------------------------------------------------------------------------------------------------
The facility FLAGSTAFF MEDICAL CENTER has better than the national benchmark in terms of Healthcare-Associated Infections. The patient survey scores are 2.00 stars, and the cleanliness is 2.10 stars.

EXECUTIVE SUMMARY
----------------------------------------------------------------------------------------------------
The facility FLAGSTAFF MEDICAL CENTER has a strong performance in Healthcare-Associated Infections. The hospital is better than the National Benchmark in this area. However, the hospital has some weaknesses in the patient survey.

CLINICAL
----------------------------------------------------------------------------------------------------
Healthcare-Associated Infections - CLABSI (ICU + select wards): 5851.0 Device Days. Cases are 0.0. The Clabsi is 0.00.

COMMUNITY REPORT
-----------------------------------------------------------------------------------

## Task 10: Save Results
--------------------------------------------------------------------------


In [187]:
# Save evaluation outputs

output_dir = Path("Data/Evaluation")
output_dir.mkdir(parents=True, exist_ok=True)

training_results.to_csv(output_dir / "training_results.csv", index=False)
training_summary.to_csv(output_dir / "training_summary.csv", index=False)
development_results.to_csv(output_dir / "development_results.csv", index=False)
development_summary.to_csv(output_dir / "development_summary.csv", index=False)
testing_results.to_csv(output_dir / "testing_results.csv", index=False)
testing_summary.to_csv(output_dir / "testing_summary.csv", index=False)
statistical_results.to_csv(output_dir / "statistical_results.csv", index=False)

print_outputs(
    "Evaluation Notebook Complete",
    ("Training Generations", len(training_results)),
    ("Development Generations", len(development_results)),
    ("Testing Generations", len(testing_results)),
    ("Output Folder", str(output_dir))
)

Evaluation Notebook Complete
Training Generations
20
--------------------------------------------------------------------------------
Development Generations
20
--------------------------------------------------------------------------------
Testing Generations
20
--------------------------------------------------------------------------------
Output Folder
Data\Evaluation
--------------------------------------------------------------------------------
